In [1]:
import torch
import torch.fft as fft
import json
import numpy as np
import os
import torch.nn as nn

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Load the configuration from the JSON file
config_path = "data_config.json"
with open(config_path, "r") as config_file:
    config = json.load(config_file)

# Access the paths from the config
data_path = config["data_path"]

def load_eeg_data(train, subject, avg):
    if train:
        file_name = 'preprocessed_eeg_training.npy'
        n_classes = 1654
        samples_per_class = 10
        repeat_times = 4
    else:
        file_name = 'preprocessed_eeg_test.npy'
        n_classes = 200
        samples_per_class = 1
        repeat_times = 80

    file_path = os.path.join(data_path, subject, file_name)
    data = np.load(file_path, allow_pickle=True)

    preprocessed_eeg_data = torch.from_numpy(data['preprocessed_eeg_data']).float().detach()
    times = torch.from_numpy(data['times']).detach()[20:]
    ch_names = data['ch_names']
    #self.times = times
    #self.ch_names = ch_names

    eeg_data_list = []
    label_list = []
    for i in range(n_classes):
        start_index = i * samples_per_class
        eeg_data = preprocessed_eeg_data[start_index: start_index + samples_per_class]
        if avg:
            labels = torch.full((samples_per_class,), i, dtype=torch.long).detach()
        else:
            labels = torch.full((samples_per_class * repeat_times,), i, dtype=torch.long).detach()
        if avg:
            eeg_data = torch.mean(eeg_data, 1)
        eeg_data_list.append(eeg_data)
        label_list.append(labels)
    return eeg_data_list, label_list

train_list_avg, train_label_list_avg = load_eeg_data(True, 'sub-08', True)
eeg_data = torch.cat(train_list_avg, dim=0).view(-1, *train_list_avg[0].shape[-2:])
eeg_data.shape

torch.Size([16540, 63, 100])

In [13]:
fft_eeg = fft.rfft(eeg_data, dim=-1)
fft_eeg.shape

torch.Size([16540, 63, 51])

In [14]:
magnitudes = fft_eeg.abs()
spectrum = magnitudes[:,:,1:]
spectrum.shape

torch.Size([16540, 63, 50])

In [23]:
phase = fft_eeg.angle()
phase.shape

torch.Size([16540, 63, 51])

In [20]:
power = spectrum ** 2
amp = 2 * torch.sqrt(torch.sum(power, dim=2)) #/ 100
amp.shape

torch.Size([16540, 63])

In [15]:
sampling_rate = 100
freqs_eeg = fft.rfftfreq(eeg_data.size(-1), d=1.0 / sampling_rate)
freqs_eeg.shape

torch.Size([51])

In [21]:
offset = fft_eeg.real[:, :, 0] / 100
offset.shape

torch.Size([16540, 63])

In [ ]:
def fft_band_extraction(self, x, sampling_rate):
        # 对输入的EEG信号进行快速傅里叶变换（FFT）
        fft_x = fft.rfft(x, dim=-1)
        freqs = fft.rfftfreq(x.size(-1), d=1.0 / sampling_rate)

        # 提取特定频段的频域特征（Theta, Alpha, Beta, Gamma）
        theta = (freqs >= 4) & (freqs <= 7)
        alpha = (freqs >= 8) & (freqs <= 13)
        beta = (freqs >= 14) & (freqs <= 29)
        gamma = (freqs >= 30) & (freqs <= 47)

        band_features = torch.cat([
            fft_x[:, :, theta].abs().mean(dim=-1, keepdim=True),
            fft_x[:, :, alpha].abs().mean(dim=-1, keepdim=True),
            fft_x[:, :, beta].abs().mean(dim=-1, keepdim=True),
            fft_x[:, :, gamma].abs().mean(dim=-1, keepdim=True)
        ], dim=-1)

        return band_features

x_fft = fft_band_extraction(eeg_data, sampling_rate)
num_bands=4
conv1d = nn.Conv1d(in_channels=num_bands, out_channels=64, kernel_size=3, padding=1)
x_fft = conv1d(x_fft.transpose(1, 2))

In [3]:
import re
sub = 'sub-08'

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None

subject_id = extract_id_from_string(sub)
batch_size = 64
subject_ids = torch.full((batch_size,), subject_id, dtype=torch.long)
print(subject_ids.shape)

num_subjects=10
d_model = 124
subject_embedding = nn.Embedding(num_subjects, d_model)
subject_emb = subject_embedding(subject_ids)
print(subject_emb.shape)
print(subject_emb.unsqueeze(1).shape)

torch.Size([64])
torch.Size([64, 124])
torch.Size([64, 1, 124])


In [3]:
from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader, Dataset

# Load the configuration from the JSON file
config_path = "data_config.json"
with open(config_path, "r") as config_file:
    config = json.load(config_file)

# Access the paths from the config
data_path = config["data_path"]

train_dataset = SoloEEGDataset(data_path, subjects=['sub-08'], train=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=True)

D:\Program\anaconda3\envs\BCI\Lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


UnpicklingError: Failed to interpret file './Preprocessed_data_100Hz\\sub-08\\preprocessed_eeg_training.npy' as a pickle

In [7]:
import torch
import torch.nn as nn
import torch.fft as fft
from layers.Embed import DataEmbedding, SubjectEmbedding
from layers.Transformer_EncDec import Encoder, EncoderLayer
from layers.SelfAttention_Family import FullAttention, AttentionLayer
from model.ReVisModels_Draft import FreqEncoder, TimeEncoder, ReVisEncoder


In [15]:
from utils.Configs import Configs
import re
default_configs = Configs()
device = "cuda:0"
sub = 'sub-08'
freqEncoder = FreqEncoder(default_configs)
freqEncoder.to(device)
timeEncoder = TimeEncoder(default_configs)
timeEncoder.to(device)
revisEncoder = ReVisEncoder(default_configs, encoder_type == 'semantic')
revisEncoder.to(device)

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None

with torch.no_grad():
    for batch_idx, (eeg_data, label, clip_text_features, clip_img_features, vae_img_features, text, img) in enumerate(train_loader):
        eeg_data = eeg_data.to(device)
        batch_size = eeg_data.size(0)
        print(f'batch size is {batch_size}')
        subject_id = extract_id_from_string(sub)
        subject_ids = torch.full((batch_size,), subject_id, dtype=torch.long).to(device)
        eeg_freq_features, attn = freqEncoder(eeg_data, sub_ids = subject_ids)
        eeg_freq_features = eeg_freq_features.float()
        print(f'eeg_freq_features shape is{eeg_freq_features.shape}')
        eeg_time_features = timeEncoder(eeg_data, sub_ids = subject_ids)
        eeg_time_features = eeg_time_features.float()
        print(f'eeg_time_features shape is{eeg_time_features.shape}')
        eeg_features = revisEncoder(eeg_data, sub_ids = subject_ids)
        eeg_features = eeg_features.float()
        print(f'eeg_features shape is{eeg_features.shape}')
        break
        
final = torch.cat(( eeg_freq_features, eeg_time_features), dim=-1)
print(f'final feature shape is {final.shape}')

patch_num is 24
patch_num is 24
batch size is 32
Encoderlayer input shape is: torch.Size([32, 64, 256])
Encoderlayer input shape is: torch.Size([32, 64, 256])
eeg_freq_features shape istorch.Size([32, 64, 256])
time encoder embedding input size is : torch.Size([32, 64, 24, 10])
Encoderlayer input shape is: torch.Size([2048, 24, 64])
Encoderlayer input shape is: torch.Size([2048, 24, 64])
eeg_time_out_features shape istorch.Size([32, 64, 1536])
eeg_time_out_features proj shape istorch.Size([32, 64, 256])
eeg_time_features shape istorch.Size([32, 64, 256])
Encoderlayer input shape is: torch.Size([32, 64, 256])
Encoderlayer input shape is: torch.Size([32, 64, 256])
time encoder embedding input size is : torch.Size([32, 64, 24, 10])
Encoderlayer input shape is: torch.Size([2048, 24, 64])
Encoderlayer input shape is: torch.Size([2048, 24, 64])
eeg_time_out_features shape istorch.Size([32, 64, 1536])
eeg_time_out_features proj shape istorch.Size([32, 64, 256])
features shape istorch.Size([32

In [9]:
a = torch.randn((100, 250))
b = torch.randn((32* 64, 24, 250))
(a+b).shape

torch.Size([2048, 24, 250])

In [11]:
a = torch.randn((100, 64))
a = a.view(100, 4, -1)

In [13]:
flatten = nn.Flatten(start_dim=-2)
flatten.requires_grad_(False)
a = torch.randn((100, 64, 10, 63))
b = flatten(a[:,0,:,:])
print(b.shape)
c = flatten(a)
print(c.shape)

torch.Size([100, 630])
torch.Size([100, 64, 630])


In [47]:
conv = nn.Conv2d(1, 40, (1, 25), stride=1)
conv.requires_grad_(False)
d = torch.randn((32, 1, 2, 26))
print(conv(d).shape)

torch.Size([32, 40, 2, 2])


In [54]:
conv = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=1)
conv.requires_grad_(False)
d = torch.randn((32, 64, 500))
print(conv(d).shape)
print(conv.state_dict()['weight'].shape)

torch.Size([32, 128, 500])
torch.Size([128, 64, 1])


In [7]:
import os
output_dir = "./outputs/"
encoder_type = 'semantic'
save_model = os.path.join(output_dir, 'model', encoder_type)
print (save_model)

./outputs/model\semantic


In [5]:
batch_eeg_data = eeg_data[:32]
batch_eeg_data.shape

torch.Size([32, 63, 100])

In [10]:
from model.ReVisModels import ReVisEncoder
from utils.Configs import Configs
import re

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None

device = "cuda:0"
sub = 'sub-08'
encoder_type = 'ReVIS'
batch_size = 32

default_configs = Configs()
testReVISEncoder = ReVisEncoder(default_configs, encoder_type=encoder_type).to(device)
testReVISEncoder.requires_grad_(False)

subject_id = extract_id_from_string(sub)
subject_ids = torch.full((batch_size,), subject_id, dtype=torch.long).to(device)
batch_eeg_data = batch_eeg_data.to(device)
hidden_states = testReVISEncoder(batch_eeg_data, sub_ids=subject_ids)
print(hidden_states.shape)
del batch_eeg_data, subject_ids, testReVISEncoder

patch_num is 24
torch.Size([32, 32768])


In [4]:
print(torch.cuda.is_available())

True
